# Heart Disease Data Science

Group project for a Data Science course.

Data: UCI Heart Disease (sections 1 to 8) and a small table of student graduation data (section 9).

## 1. Load the data

4 files, one from each hospital. Same 14 columns in every file.

| Column | Meaning |
|---|---|
| `age` | Age in years |
| `sex` | Sex (0 = female, 1 = male) |
| `cp` | Chest pain type (1 = typical angina, 2 = atypical angina, 3 = non-anginal, 4 = asymptomatic) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar above 120 mg/dl (1 = yes, 0 = no) |
| `restecg` | Resting ECG result |
| `thalach` | Maximum heart rate achieved |
| `exang` | Angina caused by exercise (1 = yes, 0 = no) |
| `oldpeak` | ST depression caused by exercise |
| `slope` | Slope of the ST segment during peak exercise |
| `ca` | Number of major vessels seen by fluoroscopy |
| `thal` | Thallium test result |
| `target` | Diagnosis (0 = no heart disease, 1 to 4 = heart disease) |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Perceptron
from sklearn.metrics import r2_score

# column names from heart-disease.names
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

# read from the data/ folder, or from GitHub if opened in Colab
DATA_DIR = 'data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'https://raw.githubusercontent.com/Kireei/Heart-Disease-Data-Science/main/data'

files = ['cleveland', 'hungarian', 'switzerland', 'va']

# missing values are written as '?' in the files
df = pd.concat(
    [pd.read_csv(f'{DATA_DIR}/processed.{name}.data', names=columns, na_values='?') for name in files],
    ignore_index=True,
)

display(df.head())
df.info()

## 2. Cleaning

In [ ]:
print(df.isnull().sum())
print('duplicate rows:', df.duplicated().sum())

In [ ]:
df_clean = df.drop_duplicates()

# ca, slope, thal: more than 30% missing
df_clean = df_clean.drop(columns=['ca', 'slope', 'thal'])

# median because chol and oldpeak have outliers
for col in ['trestbps', 'chol', 'thalach', 'oldpeak']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# categories, so use the most common value
for col in ['fbs', 'restecg', 'exang']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(df_clean.isnull().sum().sum(), 'missing values left')
print(df_clean.shape)

## 3. Exploration

In [ ]:
df_clean.describe()

Min of `trestbps` and `chol` is 0. Checked in section 5.

In [ ]:
# target 1 to 4 are all heart disease, so make it 0/1
df_clean['target_binary'] = (df_clean['target'] > 0).astype(int)

for col in ['sex', 'cp', 'fbs', 'exang', 'target_binary']:
    display(df_clean[col].value_counts().to_frame().T)

725 men, 193 women. 508 patients (55.3%) have heart disease.

In [ ]:
# the data is 79% men, so compare the rate, not the count
df_clean.groupby('sex')['target_binary'].agg(['sum', 'count', 'mean']).round(3)

Heart disease: about 63% of men, 26% of women.

## 4. Charts

In [ ]:
sns.set_theme(style='whitegrid')
fig = plt.figure(figsize=(15, 12))

# heart disease by sex
ax1 = plt.subplot(2, 2, 1)
sns.countplot(x='target_binary', hue='sex', data=df_clean, palette='viridis', ax=ax1)
ax1.set_title('Heart Disease Cases by Sex')
ax1.set_xlabel('Diagnosis (0 = Healthy, 1 = Heart Disease)')
ax1.set_ylabel('Number of Patients')
handles, _ = ax1.get_legend_handles_labels()
ax1.legend(handles, ['Female', 'Male'])

# average blood pressure for each age
ax2 = plt.subplot(2, 2, 2)
age_bp = df_clean.groupby('age')['trestbps'].mean().reset_index()
sns.lineplot(x='age', y='trestbps', data=age_bp, color='red', marker='o', ax=ax2)
ax2.set_title('Average Resting Blood Pressure by Age')
ax2.set_xlabel('Age (Years)')
ax2.set_ylabel('Blood Pressure (mm Hg)')

# age and maximum heart rate
ax3 = plt.subplot(2, 2, 3)
sns.scatterplot(x='age', y='thalach', hue='target_binary', style='target_binary',
                data=df_clean, alpha=0.7, palette='coolwarm', ax=ax3)
ax3.set_title('Age and Maximum Heart Rate')
ax3.set_xlabel('Age')
ax3.set_ylabel('Maximum Heart Rate Achieved (thalach)')

# chest pain types. value_counts sorts from the biggest, here the order is 4, 3, 2, 1
ax4 = plt.subplot(2, 2, 4)
cp_counts = df_clean['cp'].value_counts()
cp_labels = ['Asymptomatic (4)', 'Non-anginal (3)', 'Atypical angina (2)', 'Typical angina (1)']
ax4.pie(cp_counts, labels=cp_labels, autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette('pastel'), explode=(0.05, 0, 0, 0))
ax4.set_title('Share of Chest Pain Types')

plt.tight_layout()
plt.show()

- More men than women in both groups, because the data is 79% men.
- Blood pressure tends to go up with age.
- Maximum heart rate goes down with age, and is lower for patients with heart disease.
- 54% of patients have chest pain type 4 (asymptomatic = no chest pain).

## 5. Descriptive statistics

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

stats = df_clean[num_cols].describe()
stats.loc['median'] = df_clean[num_cols].median()
stats.loc['variance'] = df_clean[num_cols].var()
stats.loc['mode'] = df_clean[num_cols].mode().iloc[0]
stats

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_clean[num_cols], palette='Set3')
plt.title('Data Spread and Outliers')
plt.show()

In [ ]:
# how many zeros
(df_clean[['chol', 'trestbps']] == 0).sum()

Mean and median are close for `age` (53.5 / 54), `trestbps` (132 / 130) and `thalach` (137.7 / 140).

`chol`: mean 199.9, median 223, mode 0. 172 rows have `chol` = 0 and 1 row has `trestbps` = 0. Probably missing values written as 0. Not fixed here, so the `chol` numbers are too low.

## 6. Correlation

In [ ]:
corr = df_clean[num_cols + ['target_binary']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlGn', fmt='.2f')
plt.title('Correlation Heatmap of Heart Disease Risk Variables')
plt.show()

- `thalach` and diagnosis: -0.38
- `oldpeak` and diagnosis: 0.37
- `age` and `thalach`: -0.35

## 7. Bayes' theorem

Chance of heart disease if the chest pain is asymptomatic (`cp` = 4). "Sick" = has heart disease.

$$P(Sick \mid Asym) = \frac{P(Asym \mid Sick) \cdot P(Sick)}{P(Asym)}$$

In [ ]:
n = len(df_clean)
sick = df_clean['target_binary'] == 1
asym = df_clean['cp'] == 4

p_sick = sick.sum() / n
p_asym = asym.sum() / n
p_asym_given_sick = (sick & asym).sum() / sick.sum()

p_sick_given_asym = p_asym_given_sick * p_sick / p_asym

print(f'P(sick) = {p_sick:.4f}')
print(f'P(asym) = {p_asym:.4f}')
print(f'P(asym | sick) = {p_asym_given_sick:.4f}')
print(f'P(sick | asym) = {p_sick_given_asym:.4f}')

# count it directly to check
print((sick & asym).sum(), 'of', asym.sum(), '=', round((sick & asym).sum() / asym.sum(), 4))

About 79%. The direct count gives the same result (392 of 496).

## 8. Simple linear regression

Predict `oldpeak` from `age`.

In [ ]:
# remove rows with negative oldpeak
df_reg = df_clean[df_clean['oldpeak'] >= 0]
X = df_reg[['age']]
y = df_reg['oldpeak']

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_reg, x='age', y='oldpeak', alpha=0.7)
plt.title('Age and Oldpeak (without negative values)')
plt.xlabel('Age')
plt.ylabel('Oldpeak')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

No clear line. Many points are at 0, and the values are more spread out for older patients.

In [ ]:
reg = LinearRegression().fit(X, y)
print(f'oldpeak = {reg.intercept_:.3f} + {reg.coef_[0]:.3f} * age')

ages = pd.DataFrame({'age': [50, 60]})
print('age 50 and 60:', reg.predict(ages).round(2))

# same data as training, no train/test split
print('R-squared:', round(r2_score(y, reg.predict(X)), 2))

R-squared is 0.06, so age only explains about 6% of `oldpeak`. The prediction is 0.79 at age 50 and 1.05 at age 60. Age alone is not enough.

## 9. Decision tree and perceptron

Different data: 15 students. Predict on time or late.

| No | Gender | Student Status | Marital Status | GPA (Semester 1-6) | Graduation |
|---|---|---|---|---|---|
| 1 | Male | Student only | Not married | 3.17 | On time |
| 2 | Male | Working | Not married | 3.30 | On time |
| 3 | Female | Student only | Not married | 3.01 | On time |
| 4 | Female | Student only | Married | 3.25 | On time |
| 5 | Male | Working | Married | 3.20 | On time |
| 6 | Male | Working | Married | 2.50 | Late |
| 7 | Female | Working | Married | 3.00 | Late |
| 8 | Female | Working | Not married | 2.70 | Late |
| 9 | Male | Working | Not married | 2.40 | Late |
| 10 | Female | Student only | Married | 2.50 | Late |
| 11 | Female | Student only | Not married | 2.50 | Late |
| 12 | Female | Student only | Not married | 3.50 | On time |
| 13 | Male | Working | Married | 3.30 | On time |
| 14 | Male | Student only | Married | 3.25 | On time |
| 15 | Male | Student only | Not married | 2.30 | Late |

New students:

| No | Gender | Student Status | Marital Status | GPA (Semester 1-6) |
|---|---|---|---|---|
| 1 | Male | Student only | Not married | 2.50 |
| 2 | Female | Working | Not married | 2.50 |

In [ ]:
# gender: male 0, female 1 | status: student only 0, working 1 | married: no 0, yes 1 | GPA
X_train = np.array([
    [0, 0, 0, 3.17], [0, 1, 0, 3.30], [1, 0, 0, 3.01], [1, 0, 1, 3.25],
    [0, 1, 1, 3.20], [0, 1, 1, 2.50], [1, 1, 1, 3.00], [1, 1, 0, 2.70],
    [0, 1, 0, 2.40], [1, 0, 1, 2.50], [1, 0, 0, 2.50], [1, 0, 0, 3.50],
    [0, 1, 1, 3.30], [0, 0, 1, 3.25], [0, 0, 0, 2.30]
])

# on time 1, late 0
y_train = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0])

X_new = np.array([
    [0, 0, 0, 2.50],
    [1, 1, 0, 2.50]
])

### Decision trees

Made by hand from the table.

**Tree 1**: GPA above 3.00 is on time, 3.00 or below is late.

**Tree 2**:

- GPA 3.00 or below: late
- GPA above 3.50: on time
- GPA from 3.01 to 3.50:
  - student only: on time
  - working and not married: on time
  - working and married: late

In [ ]:
def tree_1(gender, working, married, gpa):
    return 1 if gpa > 3.00 else 0


def tree_2(gender, working, married, gpa):
    if gpa <= 3.00:
        return 0
    if gpa > 3.50:
        return 1
    if working == 0:
        return 1
    return 0 if married == 1 else 1


for name, tree in [('Tree 1', tree_1), ('Tree 2', tree_2)]:
    pred = np.array([tree(*row) for row in X_train])
    print(name, '-', (pred == y_train).sum(), 'of 15 correct, wrong rows:', np.where(pred != y_train)[0] + 1)
    print('  new students:', [tree(*row) for row in X_new])

- Tree 1: 15 of 15.
- Tree 2: 13 of 15. Wrong on rows 5 and 13. Both are working and married (GPA 3.20 and 3.30) but graduated on time, so the "working and married = late" rule does not match the table.
- New students: late for both, in both trees.

### Perceptron

In [ ]:
clf = Perceptron(tol=1e-3, random_state=42)
clf.fit(X_train, y_train)

pred = clf.predict(X_train)
print('new students:', clf.predict(X_new))
print('weights:', clf.coef_[0].round(2), 'bias:', clf.intercept_[0])
print('epochs:', clf.n_iter_)
print((pred == y_train).sum(), 'of 15 correct, wrong rows:', np.where(pred != y_train)[0] + 1)
print('rows predicted late:', np.where(pred == 0)[0] + 1)

- New students: on time for student 1, late for student 2. Different from the trees for student 1.
- 10 of 15 correct. It says late only for rows 7 and 8, but 7 students are late. Guessing on time for everyone already gives 8 of 15.
- Weights: gender -5, working -6, married 0, GPA about 4.2, bias -4. Student 1: 4.2 x 2.5 - 4 = about 6.5, so on time.
- It stopped after 7 epochs (`tol=1e-3`) and GPA is not scaled. Tree 1 gets all 15 rows with GPA only, so this result is not reliable yet.

## 10. Summary

- 918 rows after cleaning. `chol` still has 172 zeros.
- Heart disease: 63% of men, 26% of women, 79% of patients with asymptomatic chest pain.
- Lower `thalach` and higher `oldpeak` go with heart disease (correlation below 0.4).
- Age alone cannot predict `oldpeak` (R-squared 0.06).
- Student table: Tree 1 gets 15 of 15, Tree 2 gets 13, the perceptron gets 10.